<a href="https://colab.research.google.com/github/dapoajike-cyber/RTLLM-Verilog-Generation-Experiment/blob/main/RTLLM_Verilog_Generation_Experiment_Qwen32b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RTLLM Verilog Generation Experiment

This notebook walks through a complete experiment for generating RTL Verilog from RTLLM design descriptions with a large language model API. It is designed for course use and can be used to build a feedback-driven RTL generation agent.

## Learning goals
1. Use a large language model API from Python.
2. Install Verilog tooling for local checks.
3. Download and inspect the RTLLM dataset.
4. Generate Verilog from `design_description.txt`.
5. Run `iverilog` and `vvp` to check syntax and basic simulation behavior.
6. Build a more advanced feedback loop that retries after compiler or simulation failures.


## Submission requirements

When you submit this notebook, **keep the cell outputs visible**. Do not clear outputs before submission.

Your submission should preserve at least these outputs:
- environment and API-key checks
- software/tool installation verification
- selected RTLLM design path and printed specification
- generated Verilog output or saved-file confirmation
- `iverilog` compile result
- `vvp` simulation result
- final comparison table for basic vs. advanced methods

Before submitting, re-run the important cells from top to bottom and confirm that the outputs remain visible for grading.


## Important note about tools

This notebook focuses on a **portable local workflow** built around `iverilog` and `vvp`. That makes it practical for class use even when students do not have access to proprietary EDA tools.

The workflow is therefore organized into two layers:
- **Layer 1: LLM API usage** — generate Verilog from RTLLM specs.
- **Layer 2: local Verilog verification** — compile with `iverilog` and run with `vvp`.

This is a practical classroom workflow for checking syntax and basic testbench behavior.


## Step 0: Before you begin

You will need:
- Python 3.10+
- `git`
- Jupyter Notebook or JupyterLab
- An NVIDIA API key
- `iverilog`
- `vvp`

Recommended environment variables:
- `NVIDIA_API_KEY`
- `NVIDIA_MODEL`

This notebook uses NVIDIA's officially documented OpenAI-compatible hosted endpoint at `https://integrate.api.nvidia.com/v1`.

Do **not** paste API keys directly into notebook cells. Set them in your shell before starting Jupyter.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import textwrap
from pathlib import Path

WORKDIR = Path.cwd()
RTLLM_DIR = WORKDIR / 'RTLLM'
GENERATIONS_DIR = WORKDIR / 'generated_rtl'
GENERATIONS_DIR.mkdir(exist_ok=True)

NVIDIA_BASE_URL = 'https://integrate.api.nvidia.com/v1'
DEFAULT_NVIDIA_MODEL = 'qwen/qwen2.5-coder-32b-instruct'

print(f'Working directory: {WORKDIR}')
print(f'RTLLM directory:   {RTLLM_DIR}')
print(f'Outputs directory: {GENERATIONS_DIR}')
print(f'NVIDIA base URL:   {NVIDIA_BASE_URL}')

for key in ['NVIDIA_API_KEY', 'NVIDIA_MODEL']:
    print(f'{key}:', 'SET' if os.getenv(key) else 'MISSING')

if not os.getenv('NVIDIA_MODEL'):
    print(f'NVIDIA_MODEL is not set; the notebook will default to {DEFAULT_NVIDIA_MODEL}.')


Working directory: /content
RTLLM directory:   /content/RTLLM
Outputs directory: /content/generated_rtl
NVIDIA base URL:   https://integrate.api.nvidia.com/v1
NVIDIA_API_KEY: MISSING
NVIDIA_MODEL: MISSING
NVIDIA_MODEL is not set; the notebook will default to qwen/qwen2.5-coder-32b-instruct.


## Step 1: Install the software and verify the Python environment

Use the code cells below as executable setup references. If your environment is already prepared, you can still run the verification cell and keep its output for submission.


### Executable setup commands

Use the reference cells below that match your environment. These cells are meant to be copied and adapted as needed, and their output can be preserved in the submitted notebook.


In [ ]:
# Linux / Ubuntu / Debian setup reference (with sudo)
print('sudo apt update')
print('sudo apt install -y python3 python3-pip python3-venv git iverilog')
print('python3 -m pip install --upgrade pip notebook openai tqdm scipy pandas')


sudo apt update
sudo apt install -y python3 python3-pip python3-venv git iverilog
python3 -m pip install --upgrade pip notebook openai tqdm scipy pandas


In [ ]:
# Linux setup reference (no sudo access)
print('python3 -m venv ~/rtllm-env')
print('source ~/rtllm-env/bin/activate')
print('python -m pip install --upgrade pip notebook openai tqdm scipy pandas')
print('# Use a preinstalled iverilog/vvp if your course cluster already provides them, or add your local install to PATH.')
print('export PATH=~/bin:$PATH')
print('which iverilog')
print('which vvp')


python3 -m venv ~/rtllm-env
source ~/rtllm-env/bin/activate
python -m pip install --upgrade pip notebook openai tqdm scipy pandas
# Use a preinstalled iverilog/vvp if your course cluster already provides them, or add your local install to PATH.
export PATH=~/bin:$PATH
which iverilog
which vvp


In [ ]:
# macOS / Homebrew setup reference
print('brew install python git icarus-verilog')
print('python3 -m pip install --upgrade pip notebook openai tqdm scipy pandas')


brew install python git icarus-verilog
python3 -m pip install --upgrade pip notebook openai tqdm scipy pandas


In [ ]:
# Windows setup reference (PowerShell or Command Prompt)
print('Install Python from https://www.python.org/downloads/ or a course-approved distribution.')
print('Install Icarus Verilog from https://bleyer.org/icarus/ or a course-approved package source.')
print('py -m pip install --upgrade pip notebook openai tqdm scipy pandas')
print('After installation, make sure iverilog.exe and vvp.exe are available on PATH.')


Install Python from https://www.python.org/downloads/ or a course-approved distribution.
Install Icarus Verilog from https://bleyer.org/icarus/ or a course-approved package source.
py -m pip install --upgrade pip notebook openai tqdm scipy pandas
After installation, make sure iverilog.exe and vvp.exe are available on PATH.


### Environment variable setup references

Set your NVIDIA API credentials before launching Jupyter. Examples:


In [ ]:
# POSIX shell example (Linux / macOS)
print("export NVIDIA_API_KEY='nvapi-...'")
print("export NVIDIA_MODEL='qwen/qwen2.5-coder-32b-instruct'")
print('jupyter notebook')


export NVIDIA_API_KEY='nvapi-...'
export NVIDIA_MODEL='qwen/qwen2.5-coder-32b-instruct'
jupyter notebook


In [ ]:
# Windows PowerShell example
print("$env:NVIDIA_API_KEY='nvapi-...'")
print("$env:NVIDIA_MODEL='qwen/qwen2.5-coder-32b-instruct'")
print('jupyter notebook')


$env:NVIDIA_API_KEY='nvapi-...'
$env:NVIDIA_MODEL='qwen/qwen2.5-coder-32b-instruct'
jupyter notebook


In [ ]:
# Windows Command Prompt example
print('set NVIDIA_API_KEY=nvapi-...')
print('set NVIDIA_MODEL=qwen/qwen2.5-coder-32b-instruct')
print('jupyter notebook')


set NVIDIA_API_KEY=nvapi-...
set NVIDIA_MODEL=qwen/qwen2.5-coder-32b-instruct
jupyter notebook


In [ ]:
import importlib.util
tools = {
    'python3': shutil.which('python3') or shutil.which('python') or shutil.which('py'),
    'iverilog': shutil.which('iverilog'),
    'vvp': shutil.which('vvp'),
}
packages = ['openai', 'pandas', 'tqdm', 'scipy']
print('Tool checks:')
for name, path in tools.items():
    print(f'  {name}:', path if path else 'MISSING')
print('Package checks:')
for pkg in packages:
    print(f'  {pkg}:', 'OK' if importlib.util.find_spec(pkg) else 'MISSING')

python_bin = shutil.which('python3') or shutil.which('python')
if python_bin:
    subprocess.run([python_bin, '--version'], check=False)
elif shutil.which('py'):
    subprocess.run(['py', '--version'], check=False)
if tools['iverilog']:
    subprocess.run([tools['iverilog'], '-V'], check=False)
if tools['vvp']:
    subprocess.run([tools['vvp'], '-V'], check=False)


Tool checks:
  python3: /usr/bin/python3
  iverilog: MISSING
  vvp: MISSING
Package checks:
  openai: OK
  pandas: OK
  tqdm: OK
  scipy: OK


### Expected output guidance for setup

For a healthy setup, you should see:
- a valid path for Python, `iverilog`, and `vvp`
- `OK` for Python packages such as `openai` and `pandas`
- version text printed for Python and the Verilog tools

Windows users and Linux users without `sudo` should run the same verification cell and confirm that the paths and package checks succeed in their environment.
Keep this output visible in the submitted notebook.


### Notebook-native setup commands

If students are working directly inside Jupyter, they can also use notebook-native commands such as `!pip` and `!` shell commands. These examples are useful when they do not want to switch back and forth between the notebook and a terminal.


In [ ]:
# Example notebook-native package installation commands
print('!pip install --upgrade pip')
print('!pip install notebook openai tqdm scipy pandas')
print('!python -m pip install --upgrade pip')
print('!python -m pip install notebook openai tqdm scipy pandas')


!pip install --upgrade pip
!pip install notebook openai tqdm scipy pandas
!python -m pip install --upgrade pip
!python -m pip install notebook openai tqdm scipy pandas


In [ ]:
# Example notebook-native shell commands for quick checks
print('!python --version')
print('!iverilog -V')
print('!vvp -V')
print('!where python   # Windows')
print('!where iverilog # Windows')
print('!which python   # Linux/macOS')
print('!which iverilog # Linux/macOS')


!python --version
!iverilog -V
!vvp -V
!where python   # Windows
!where iverilog # Windows
!which python   # Linux/macOS
!which iverilog # Linux/macOS


## Step 2: Use the NVIDIA large language model API

This notebook uses NVIDIA's officially documented OpenAI-compatible Chat Completions API:
- Base URL: `https://integrate.api.nvidia.com/v1`
- Endpoint: `POST /chat/completions`
- Auth: `Authorization: Bearer $NVIDIA_API_KEY`

The key idea is simple:
1. Read the RTLLM design spec.
2. Ask the model for **only Verilog code**.
3. Save the result to a `.v` file.

The prompt should be strict about module name, I/O names, widths, and output formatting.


In [ ]:
from openai import OpenAI

def strip_code_fences(text: str) -> str:
    text = text.strip()
    match = re.search(r'```(?:verilog)?\n(.*)```', text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text

def make_nvidia_client() -> OpenAI:
    api_key = os.getenv('NVIDIA_API_KEY')
    if not api_key:
        raise RuntimeError('NVIDIA_API_KEY is not set. Export it before running the notebook.')
    # TODO: finish the NVIDIA client configuration using the provided base URL and API key.
    return OpenAI(
        base_url=NVIDIA_BASE_URL,
        api_key=api_key,
    )

def get_nvidia_model() -> str:
    return os.getenv('NVIDIA_MODEL', DEFAULT_NVIDIA_MODEL)

def call_llm_for_verilog(spec_text: str, temperature: float = 0.2, max_tokens: int = 2048) -> str:
    client = make_nvidia_client()
    model = get_nvidia_model()

    system_prompt = (
        'You are a professional RTL designer. Return only synthesizable Verilog code. '
        'Do not include markdown fences. Preserve the exact module name and exact I/O names from the specification.'
    )

    # TODO: build the user prompt from the RTLLM specification. Keep the exact module name, I/O names, and widths.
    user_prompt = f'''Generate Verilog for the following RTL design:
{spec_text}
'''

    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=temperature,
        top_p=0.7,
        max_tokens=max_tokens,
        stream=False,
    )

    text = response.choices[0].message.content
    return strip_code_fences(text)


### TODO guidance for the NVIDIA API cell

Complete two things before running the helper cell above:
1. Replace `TODO_BASE_URL` with the NVIDIA hosted API base URL.
2. Replace `TODO_PROMPT` with a prompt built from `spec_text`.

Hint: the prompt should ask for **only Verilog code** and should preserve the exact RTLLM interface.


### Example shell setup for NVIDIA

Before launching Jupyter, set your environment like this:
```bash
export NVIDIA_API_KEY='nvapi-...'
export NVIDIA_MODEL='qwen/qwen2.5-coder-32b-instruct'
jupyter notebook
```

If your course staff has approved a different NVIDIA-hosted model, replace `NVIDIA_MODEL` with that model name.


In [ ]:
# TODO: Run this cell after you complete the API helper above.
print('Model selected:', get_nvidia_model())
print('NVIDIA client ready:', 'yes' if os.getenv('NVIDIA_API_KEY') else 'no')


Model selected: qwen/qwen2.5-coder-32b-instruct
NVIDIA client ready: no


### Expected output guidance for API setup

After the API readiness cell runs, you should see:
- a selected model name
- `NVIDIA client ready: yes` if your environment variable is set

Keep this output visible for submission.

TODO notebook reminder: after you complete this step, leave the finished cell output visible for grading.


## Step 3: Download RTLLM

The official repository is: `https://github.com/hkust-zhiyao/RTLLM`

RTLLM v2.0 contains 50 designs organized into categories such as Arithmetic, Memory, Control, and Miscellaneous. Each design folder contains a `design_description.txt` file and a `testbench.v`.


In [ ]:
def run(cmd, cwd=None, check=True):
    print('\n$ ' + ' '.join(cmd))
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed

if not RTLLM_DIR.exists():
    run(['git', 'clone', 'https://github.com/hkust-zhiyao/RTLLM.git', str(RTLLM_DIR)])
else:
    print('RTLLM already exists; skipping clone.')



$ git clone https://github.com/hkust-zhiyao/RTLLM.git /content/RTLLM
Cloning into '/content/RTLLM'...



In [ ]:
spec_paths = sorted(RTLLM_DIR.glob('**/design_description.txt'))
print(f'Found {len(spec_paths)} design descriptions.')
for path in spec_paths[:10]:
    print(path.relative_to(RTLLM_DIR))


Found 50 design descriptions.
Arithmetic/Accumulator/accu/design_description.txt
Arithmetic/Adder/adder_16bit/design_description.txt
Arithmetic/Adder/adder_32bit/design_description.txt
Arithmetic/Adder/adder_8bit/design_description.txt
Arithmetic/Adder/adder_bcd/design_description.txt
Arithmetic/Adder/adder_pipe_64bit/design_description.txt
Arithmetic/Comparator/comparator_3bit/design_description.txt
Arithmetic/Comparator/comparator_4bit/design_description.txt
Arithmetic/Divider/div_16bit/design_description.txt
Arithmetic/Divider/radix2_div/design_description.txt


## Step 4: Pick one RTLLM design and inspect the spec

Start with one design before attempting the full dataset. A small design makes debugging easier. Good first choices include simple adders, counters, or comparators.

The spec file tells you exactly what the model must implement. Your job is to preserve those constraints in the prompt and in the generated code.


In [ ]:
# TODO: choose one RTLLM design description file. Start with adder_8bit unless your instructor says otherwise.
DESIGN_SPEC_PATH = RTLLM_DIR / 'Arithmetic' / 'Adder' /'adder_8bit' / 'design_description.txt'
spec_text = DESIGN_SPEC_PATH.read_text()
print(DESIGN_SPEC_PATH)
print()
print(spec_text)


/content/RTLLM/Arithmetic/Adder/adder_8bit/design_description.txt

Please act as a professional verilog designer.

Implement a module of an 8-bit adder with multiple bit-level adders in combinational logic. 

Module name:  
    adder_8bit               
Input ports:
    a[7:0]: 8-bit input operand A.
    b[7:0]: 8-bit input operand B.
    cin: Carry-in input.
Output ports:
    sum[7:0]: 8-bit output representing the sum of A and B.
    cout: Carry-out output.

Implementation:
The module utilizes a series of bit-level adders (full adders) to perform the addition operation.

Give me the complete code.
 


### Expected output guidance for design selection

You should see the selected design path and the full `design_description.txt` contents.
This output proves which RTLLM task you worked on, so keep it visible in your submission.


## Step 5: Generate Verilog from the spec

This cell calls the model and prints the returned RTL.

Recommended baseline settings:
- Temperature around `0.0` to `0.3` for deterministic behavior.
- Ask for **code only**.
- Keep the exact RTLLM module name and port list.

If the model returns commentary, strip it before saving the file.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
    print('NVIDIA_API_KEY loaded from Colab Secrets.')
except Exception:
    os.environ['NVIDIA_API_KEY'] = 'nvapi-PASTE-YOUR-KEY-HERE'
    print('⚠️  Using hardcoded key.')

os.environ.setdefault('NVIDIA_MODEL', 'qwen/qwen2.5-coder-32b-instruct')
print('Key set:', 'yes' if os.getenv('NVIDIA_API_KEY') else 'no')

NVIDIA_API_KEY loaded from Colab Secrets.
Key set: yes


In [ ]:
# TODO: generate Verilog from spec_text using the helper function.
generated_verilog = call_llm_for_verilog(spec_text)
print(generated_verilog)


module adder_8bit (
    input [7:0] a,
    input [7:0] b,
    input cin,
    output [7:0] sum,
    output cout
);

    wire [7:0] carry;

    // Full adder for the least significant bit
    full_adder fa0 (
        .a(a[0]),
        .b(b[0]),
        .cin(cin),
        .sum(sum[0]),
        .cout(carry[0])
    );

    // Full adders for the remaining bits
    genvar i;
    generate
        for (i = 1; i < 8; i = i + 1) begin : fa_gen
            full_adder fa (
                .a(a[i]),
                .b(b[i]),
                .cin(carry[i-1]),
                .sum(sum[i]),
                .cout(carry[i])
            );
        end
    endgenerate

    // Assign the final carry-out
    assign cout = carry[7];

endmodule

// Full adder module definition
module full_adder (
    input a,
    input b,
    input cin,
    output sum,
    output cout
);

    assign sum = a ^ b ^ cin;
    assign cout = (a & b) | (cin & (a ^ b));

endmodule


### Expected output guidance for generated RTL

You should see Verilog text for the target module.
At minimum, the output should show the module name and ports that match the RTLLM specification.
Keep either this printed RTL or the saved-file confirmation visible in your submission.


In [ ]:
design_dir = DESIGN_SPEC_PATH.parent
design_name = design_dir.name
# TODO: save the generated Verilog to <design_name>.v inside the design directory.
output_verilog_path =  design_dir / f'{design_name}.v'
output_verilog_path.write_text(generated_verilog)
print(f'Saved generated RTL to: {output_verilog_path}')


Saved generated RTL to: /content/RTLLM/Arithmetic/Adder/adder_8bit/adder_8bit.v


### Expected output guidance for saved RTL

You should see a valid file path ending in `<design_name>.v`.
This confirms the notebook saved the model output where the later compile step expects it.


## Step 6: Compile and run the generated design locally

This is the main classroom verification path in this notebook. Compile the generated RTL and the RTLLM testbench with `iverilog`, then run the simulation with `vvp`.

This gives students a real, executable check instead of relying on the model's self-assessment.


In [ ]:
iverilog_path = shutil.which('iverilog')
vvp_path = shutil.which('vvp')
if not iverilog_path or not vvp_path:
    print('iverilog and/or vvp are not installed or not on PATH; skipping local verification.')
else:
    tb_path = design_dir / 'testbench.v'
    sim_output = design_dir / 'sim.out'
    # TODO: complete the iverilog compile command.
    compile_result = run(TODO_COMPILE_CMD, cwd=design_dir, check=False)
    print('--- Compile return code ---')
    print(compile_result.returncode)
    if compile_result.returncode == 0:
        # TODO: complete the vvp run command.
        sim_result = run(TODO_RUN_CMD, cwd=design_dir, check=False)
        print('--- Simulation return code ---')
        print(sim_result.returncode)


iverilog and/or vvp are not installed or not on PATH; skipping local verification.


### Expected output guidance for compile and simulation

For a successful run, you want to see:
- compile return code `0`
- simulation return code `0`
- printed testbench output indicating whether the design passed or failed

These outputs are grading evidence and should remain visible in the submitted notebook.


## Step 7: Interpret the local verification result

There are two useful signals in this local workflow:
- **Compile success**: `iverilog` exits successfully, so the generated Verilog and testbench elaborate cleanly.
- **Simulation behavior**: `vvp` runs the compiled design, and the printed output tells you whether the design behaved as expected for the provided testbench.

A design can compile successfully and still behave incorrectly, so always look at the simulation output and not just the compile step.


In [ ]:
print('This notebook now uses the iverilog/vvp path above as the main executable verification workflow.')


This notebook now uses the iverilog/vvp path above as the main executable verification workflow.


## Step 8: Scale up to multiple designs or multiple samples

Once one design works, students can repeat the same pattern across more RTLLM tasks:
1. Read a `design_description.txt`.
2. Generate Verilog with the NVIDIA API.
3. Save the module as `<design_name>.v`.
4. Compile with `iverilog`.
5. Run with `vvp`.
6. Record whether the design compiled and whether the simulation output looked correct.

This produces a simple classroom metric set without requiring proprietary tooling.


## Step 9: Organize repeated experiments

For classroom experiments, it is helpful to generate multiple samples per design and save them in a consistent folder structure such as:
```text
generated_rtl/
  t1/adder_8bit.v
  t1/adder_16bit.v
  ...
  t2/adder_8bit.v
  ...
```

That makes it easy to compare one-shot generation versus repeated sampling. In an `iverilog`-based class workflow, students can loop over these files and record compile success plus simulation behavior for each sample.


In [ ]:
def generate_sample_for_one_design(spec_path: Path, sample_id: int, temperature: float = 0.2):
    spec = spec_path.read_text()
    design_name = spec_path.parent.name
    sample_dir = GENERATIONS_DIR / f't{sample_id}'
    sample_dir.mkdir(parents=True, exist_ok=True)
    # TODO: call the model helper and save the result in sample_dir / f'{design_name}.v'
    rtl = call_llm_for_verilog(spec, temperature=temperature)
    out_path = sample_dir / f'{design_name}.v'
    out_path.write_text(rtl)
    return out_path

example_out = generate_sample_for_one_design(DESIGN_SPEC_PATH, sample_id=1, temperature=0.2)
print(f'Wrote sample to: {example_out}')


Wrote sample to: /content/generated_rtl/t1/adder_8bit.v


In [ ]:
def summarize_generated_sample(sample_path: Path):
    print('Sample file:', sample_path)
    print('Exists:', sample_path.exists())
    if sample_path.exists():
        print('Lines:', len(sample_path.read_text().splitlines()))

summarize_generated_sample(example_out)


Sample file: /content/generated_rtl/t1/adder_8bit.v
Exists: True
Lines: 51


### Suggested classroom metrics

For each sample or design, record at least:
- Did `iverilog` succeed?
- Did `vvp` run successfully?
- Did the output indicate correct behavior for the tested cases?

From that, students can compare single-sample results against multi-sample results and study how prompt changes affect correctness.


## Step 10: Advanced workflow — build a feedback loop for an RTL generation agent

More advanced students should not stop at one-shot generation. A stronger workflow is to use **tool feedback** to guide the next model attempt.

There are two useful advanced patterns:
1. **Compile-feedback loop** — use `iverilog` errors or elaboration failures to repair syntax/structural issues.
2. **Simulation-feedback loop** — use `vvp` output or testbench failure messages to repair behavioral issues after compilation succeeds.

A typical loop is:
1. Read `design_description.txt`.
2. Generate Verilog.
3. Run `iverilog`.
4. If compilation succeeds, run `vvp`.
5. Capture the concrete tool output.
6. Feed the original spec and the processed tool feedback back to the model.
7. Ask for a corrected version.
8. Repeat for a small, fixed number of iterations.

Important hint: tool logs can be **very long**. In practice, you often need to trim them before sending them to the model. Good strategies include:
- keep only the most relevant error lines
- keep the first and last important sections
- extract failing test names or mismatch summaries
- cap the number of characters sent to the API

This prevents the prompt from being flooded by unhelpful log details.


### Two advanced feedback examples

**Example A: compile-feedback repair**
- Run `iverilog`
- If the compile step fails, collect the important error lines
- Ask the model to fix the Verilog using the original spec plus the compile log summary

**Example B: simulation-feedback repair**
- Run `iverilog` and then `vvp`
- If simulation runs but the testbench reports failures, collect the failing messages or mismatch summary
- Ask the model to fix the logic using the original spec plus the simulation log summary

In both cases, students should avoid dumping extremely long logs into the prompt without filtering them first.


In [ ]:
def truncate_tool_feedback(text: str, max_chars: int = 2000) -> str:
    text = text.strip()
    # TODO: if the log is too long, keep only the most useful parts before sending it to the model.
    return TODO_TRUNCATED_TEXT

def call_llm_with_feedback(spec_text: str, previous_code: str, tool_feedback: str, temperature: float = 0.1, max_tokens: int = 2048) -> str:
    client = make_nvidia_client()
    model = get_nvidia_model()
    tool_feedback = truncate_tool_feedback(tool_feedback)

    # TODO: build a repair prompt that includes the original spec, the previous code, and the processed tool feedback.
    messages = TODO_MESSAGES

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        top_p=0.7,
        max_tokens=max_tokens,
        stream=False,
    )
    return strip_code_fences(response.choices[0].message.content)

def try_iverilog_compile(verilog_path: Path, testbench_path: Path):
    iverilog_path = shutil.which('iverilog')
    if not iverilog_path:
        return False, 'iverilog not available'
    result = subprocess.run([iv, '-g2012', '-o', str(verilog_path.with_suffix('.out')),
         str(verilog_path), str(testbench_path)],
        text=True,
        capture_output=True,
        cwd=verilog_path.parent,
    )
    ok = result.returncode == 0
    feedback = (result.stdout or '') + '\n' + (result.stderr or '')
    return ok, feedback.strip()

def try_vvp_run(verilog_path: Path):
    vvp_path = shutil.which('vvp')
    if not vvp_path:
        return False, 'vvp not available'
    result = subprocess.run(
        [iv, '-g2012', '-o', str(verilog_path.with_suffix('.out')),
         str(verilog_path), str(testbench_path)],
        text=True,
        capture_output=True,
        cwd=verilog_path.parent,
    )
    feedback = (result.stdout or '') + '\n' + (result.stderr or '')
    passed = TODO_PASS_CONDITION
    return passed, feedback.strip()

def feedback_loop(spec_path: Path, max_iters: int = 3):
    spec = spec_path.read_text()
    design_dir = spec_path.parent
    design_name = design_dir.name
    rtl_path = design_dir / f'{design_name}.v'
    tb_path = design_dir / 'testbench.v'

    history = []
    current_code = call_llm_for_verilog(spec)

    for iteration in range(1, max_iters + 1):
        rtl_path.write_text(current_code)
        compile_ok, compile_feedback = try_iverilog_compile(rtl_path, tb_path)
        history.append({'iteration': iteration, 'phase': 'compile', 'ok': compile_ok, 'feedback': compile_feedback[:500]})
        if not compile_ok:
            # TODO: repair from compile feedback and continue.
            current_code = TODO_COMPILE_REPAIR
            continue

        sim_ok, sim_feedback = try_vvp_run(rtl_path)
        history.append({'iteration': iteration, 'phase': 'simulation', 'ok': sim_ok, 'feedback': sim_feedback[:500]})
        if sim_ok:
            print(f'Iteration {iteration}: simulation passed.')
            break

        # TODO: repair from simulation feedback.
        current_code = TODO_SIM_REPAIR

    return history


## Suggested experiments

1. **One-shot vs. feedback loop**: Does compile success improve after 2 or 3 retries?
2. **Temperature sweep**: Compare `temperature = 0.0`, `0.2`, and `0.6`.
3. **Pass@k**: Generate multiple samples per design and compare `pass@1` to `pass@3` or `pass@5`.
4. **Prompt ablations**: Remove one instruction at a time and see what breaks.
5. **Syntax vs. function**: Count how many designs compile but still fail the testbench.

These experiments help students see that correctness depends on both prompt quality and disciplined evaluation.


## TODO reflection questions

Before you finish, answer these in your lab notes:
1. Which TODO was hardest, and why?
2. What kind of bug did `iverilog` or `vvp` help you catch?
3. How would you improve the prompt for better RTL on the first try?


## Final result presentation

Use the code cell below to present your final comparison between the basic method and the advanced method. Replace the example values with your real experiment results and keep the rendered table visible in your submission.


In [ ]:
import pandas as pd

# Fill in these counts with your real experiment results.
# Example: if you tested all 50 RTLLM designs, set total_designs_tested = 50.

basic_total_designs_tested = 0
basic_syntax_success_count = 0
basic_function_success_count = 0

advanced_total_designs_tested = 0
advanced_syntax_success_count = 0
advanced_function_success_count = 0

def safe_rate(success_count, total_count):
    return (success_count / total_count) if total_count else 0.0

results_df = pd.DataFrame([
    {
        'Method': 'Basic Method',
        'Total Designs Tested': basic_total_designs_tested,
        'Syntax Success Count': basic_syntax_success_count,
        'Functional Success Count': basic_function_success_count,
        'Syntax Correctness Rate': safe_rate(basic_syntax_success_count, basic_total_designs_tested),
        'Functional Verification Success Rate': safe_rate(basic_function_success_count, basic_total_designs_tested),
    },
    {
        'Method': 'Student-Developed Advanced Method',
        'Total Designs Tested': advanced_total_designs_tested,
        'Syntax Success Count': advanced_syntax_success_count,
        'Functional Success Count': advanced_function_success_count,
        'Syntax Correctness Rate': safe_rate(advanced_syntax_success_count, advanced_total_designs_tested),
        'Functional Verification Success Rate': safe_rate(advanced_function_success_count, advanced_total_designs_tested),
    },
])

styled_results = (results_df.style
    .format({
        'Syntax Correctness Rate': '{:.1%}',
        'Functional Verification Success Rate': '{:.1%}',
    })
    .hide(axis='index')
    .set_caption('Basic vs. Student-Developed Advanced Method')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'th', 'props': [('background-color', '#1f4e79'), ('color', 'white'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]},
    ])
    .background_gradient(subset=['Syntax Correctness Rate', 'Functional Verification Success Rate'], cmap='YlGn')
)

styled_results


Method,Total Designs Tested,Syntax Success Count,Functional Success Count,Syntax Correctness Rate,Functional Verification Success Rate
Basic Method,0,0,0,0.0%,0.0%
Student-Developed Advanced Method,0,0,0,0.0%,0.0%


### Expected output guidance for final results

Your final notebook should show a completed comparison table with:
- one row for the **Basic Method**
- one row for the **Student-Developed Advanced Method**
- syntax correctness rate
- functional verification success rate

Keep this table visible because it is part of the final deliverable.


## Final checklist

Before submitting results, each student team should be able to answer:
- Which model and temperature did we use?
- Which RTLLM designs did we test?
- Did the design compile?
- Did it pass the RTLLM testbench?
- If not, what was the most useful feedback signal?
- Did a retry loop improve the result?

Submission reminder:
- keep outputs visible
- keep compile and simulation evidence visible
- keep the final comparison table visible

That turns the exercise from a demo into an actual experiment.
